In [22]:
import sqlite3
import re

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> list[tuple[str, str]]:
    """
    严格保真英文分词（逐字符状态机）：
    - 拼回去与原句完全一致
    - 不合并、不丢弃、不修改任何字符
    """
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_words_table():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # ✅ 重建 words 表
    cursor.execute("DROP TABLE IF EXISTS words")
    cursor.execute("""
    CREATE TABLE words (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        verse_id TEXT NOT NULL,
        order_index INTEGER NOT NULL,
        word TEXT NOT NULL,
        type TEXT NOT NULL,
        entity_key TEXT,
        start_time REAL,
        end_time REAL,
        UNIQUE(verse_id, order_index)
    )
    """)

    cursor.execute("""
        CREATE INDEX IF NOT EXISTS idx_words_verse_id
        ON words(verse_id)
    """)

    cursor.execute("""
        SELECT id, text_en
        FROM verse
        WHERE text_en IS NOT NULL
    """)
    verses = cursor.fetchall()

    inserted = 0

    for verse_id, text_en in verses:
        tokens = segment_english_preserve(text_en)

        # ✅ 保真校验（强烈建议保留）
        reconstructed = "".join(word for word, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse_id:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        for idx, (token, token_type) in enumerate(tokens):
            cursor.execute("""
                INSERT INTO words (verse_id, order_index, word, type)
                VALUES (?, ?, ?, ?)
            """, (str(verse_id), idx, token, token_type))
            inserted += 1

    conn.commit()
    conn.close()

    print("✅ words 表已成功生成")
    print(f"   总 token 数：{inserted}")


if __name__ == "__main__":
    build_words_table()

✅ words 表已成功生成
   总 token 数：6652


In [13]:
# 清空数据

import sqlite3

DB_PATH = "db/bible.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("DELETE FROM words;")
conn.commit()
conn.close()

print("✅ words 表数据已全部清空")

✅ words 表数据已全部清空


In [21]:
# 增量更新

import sqlite3
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_words_table():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # ✅ 只取 verse 中存在，但 words 中不存在的 verse_id
    cursor.execute("""
        SELECT v.id, v.text_en
        FROM verse v
        WHERE v.text_en IS NOT NULL
          AND v.id NOT IN (
              SELECT DISTINCT w.verse_id
              FROM words w
          )
    """)

    verses = cursor.fetchall()
    print(f"🆕 需要增量拆分的 verse 数：{len(verses)}")

    inserted = 0

    for verse_id, text_en in verses:
        tokens = segment_english_preserve(text_en)

        reconstructed = "".join(word for word, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse_id:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        for idx, (token, token_type) in enumerate(tokens):
            try:
                cursor.execute("""
                    INSERT INTO words (
                        verse_id, order_index, word, type
                    ) VALUES (?, ?, ?, ?)
                """, (str(verse_id), idx, token, token_type))
                inserted += 1
            except sqlite3.IntegrityError:
                # ✅ 理论上不会进，除非并发写入
                pass

    conn.commit()
    conn.close()

    print("✅ 增量更新完成")
    print(f"   新增 token 数：{inserted}")


if __name__ == "__main__":
    build_words_table()

🆕 需要增量拆分的 verse 数：0
✅ 增量更新完成
   新增 token 数：0


In [27]:
# 从表 verse 填充表 tokens
# 增量更新（word_id = 本节中第几个 word，不含 punct/space）

import sqlite3
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_tokens_table():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT v.id, v.book_id, v.chapter, v.verse, v.text_en
        FROM verse v
        WHERE v.text_en IS NOT NULL
          AND v.id NOT IN (
              SELECT DISTINCT SUBSTR(t.id, 1, INSTR(t.id || '.', '.', 1, 4) - 1)
              FROM tokens t
          )
    """)

    verses = cursor.fetchall()
    print(f"🆕 需要增量拆分的 verse 数：{len(verses)}")

    inserted = 0

    for verse_id, book_id, chapter, verse_num, text_en in verses:
        tokens = segment_english_preserve(text_en)

        reconstructed = "".join(t for t, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        word_counter = 0  # ✅ 每节重置

        for idx, (token, token_type) in enumerate(tokens, start=1):
            token_id = f"{verse_id}.{idx}"  # ✅ Mt.1.1.1

            # ✅ 只有 word 才分配 word_id
            if token_type == "word":
                word_counter += 1
                word_id = word_counter
            else:
                word_id = None

            cursor.execute("""
                INSERT INTO tokens (
                    id,
                    book_id,
                    chapter_id,
                    verse_id,
                    token,
                    type,
                    entity_key,
                    word_id
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                token_id,
                book_id,
                chapter,
                verse_num,
                token,
                token_type,
                None,
                word_id
            ))
            inserted += 1

    conn.commit()
    conn.close()

    print("✅ tokens 表增量更新完成")
    print(f"   新增 token 数：{inserted}")


if __name__ == "__main__":
    build_tokens_table()

🆕 需要增量拆分的 verse 数：134
✅ tokens 表增量更新完成
   新增 token 数：2975


In [1]:
# 从表 verse 填充表 tokens
# 增量更新（word_id = 本节中第几个 word，不含 punct/space）

import sqlite3
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_tokens_table():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT v.id, v.book_id, v.chapter, v.verse, v.text_en
        FROM verse v
        WHERE v.text_en IS NOT NULL
          AND NOT EXISTS (
              SELECT 1
              FROM tokens t
              WHERE t.id LIKE v.id || '.%'
          )
    """)

    verses = cursor.fetchall()
    print(f"🆕 需要增量拆分的 verse 数：{len(verses)}")

    inserted = 0

    for verse, book_id, chapter, verse_num, text_en in verses:
        tokens = segment_english_preserve(text_en)

        reconstructed = "".join(t for t, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse:", verse)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        word_counter = 0

        for idx, (token, token_type) in enumerate(tokens, start=1):
            token_id = f"{verse}.{idx}"

            if token_type == "word":
                word_counter += 1
                word_id = word_counter
            else:
                word_id = None

            cursor.execute("""
                INSERT INTO tokens (
                    id,
                    book_id,
                    chapter,
                    verse,
                    token_id,
                    token,
                    type,
                    entity_key,
                    word_id
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                token_id,
                book_id,
                chapter,
                verse_num,
                idx,          # ✅ token_id
                token,
                token_type,
                None,
                word_id
            ))
            inserted += 1
            
    # ✅ 填充 word_seq（整章第几个 word）
    print("🔢 正在填充 word_seq...")

    cursor.execute("""
        UPDATE tokens
        SET word_seq = (
            SELECT COUNT(*)
            FROM tokens t2
            WHERE t2.book_id = tokens.book_id
              AND t2.chapter = tokens.chapter
              AND t2.type = 'word'
              AND (t2.book_id * 1000000 + t2.chapter * 10000 + t2.verse * 100 + t2.token_id) 
                  <= (tokens.book_id * 1000000 + tokens.chapter * 10000 + tokens.verse * 100 + tokens.token_id)
        )
        WHERE type = 'word'
          AND (word_seq IS NULL OR word_seq = 0)
    """)
    
    # ✅ 生成 align_id
    print("🔗 正在生成 align_id...")

    cursor.execute("""
        UPDATE tokens
        SET align_id = (
            SELECT b.abbr_en || '.' || tokens.chapter || '..' || tokens.word_seq
            FROM book b
            WHERE b.id = tokens.book_id
        )
        WHERE type = 'word'
          AND align_id IS NULL
    """)


    conn.commit()
    conn.close()

    print("✅ tokens 表增量更新完成")
    print(f"   新增 token 数：{inserted}")


if __name__ == "__main__":
    build_tokens_table()

🆕 需要增量拆分的 verse 数：134
🔢 正在填充 word_seq...
🔗 正在生成 align_id...
✅ tokens 表增量更新完成
   新增 token 数：6652


In [2]:
import sqlite3
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    """
    按字符类型切分英文文本
    返回 [(token, type), ...]
    """
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_tokens_for_chapter(book_abbr: str, book_id: int, chapter: int):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print(f"📖 处理章节：{book_abbr} {chapter}")

    # 1️⃣ 从 verses 表查询指定 book + chapter
    cursor.execute("""
        SELECT v.id, v.verse, v.text_en
        FROM verses v
        WHERE v.book_id = ?
          AND v.chapter = ?
          AND v.text_en IS NOT NULL
        ORDER BY v.verse
    """, (book_id, chapter))

    verses = cursor.fetchall()
    print(f"  需要处理的 verse 数：{len(verses)}")

    token_rows = []

    # 2️⃣ 遍历 verse → 生成 tokens
    for verse_id, verse_num, text_en in verses:
        tokens = segment_english_preserve(text_en)

        # 校验拼接一致性
        reconstructed = "".join(t for t, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        for idx, (token, token_type) in enumerate(tokens, start=1):
            token_id = f"{book_abbr}.{chapter}.{verse_num}.{idx}"

            token_rows.append((
                token_id,
                book_id,
                chapter,
                verse_num,
                idx,
                token,
                token_type,
                None  # entity_key
            ))

    # 3️⃣ 批量插入 tokens
    cursor.executemany("""
        INSERT INTO tokens (
            id,
            book_id,
            chapter,
            verse,
            token_id,
            token,
            type,
            entity_key
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, token_rows)

    print(f"✅ 插入 token 数：{len(token_rows)}")

    # 4️⃣ 填充 word_seq（整章 word 顺序）
    print("🔢 正在填充 word_seq...")

    cursor.execute("""
        SELECT id
        FROM tokens
        WHERE book_id = ?
          AND chapter = ?
          AND type = 'word'
        ORDER BY id
    """, (book_id, chapter))

    word_tokens = cursor.fetchall()

    for seq, (token_id,) in enumerate(word_tokens, start=1):
        align_id = f"{book_abbr}.{chapter}..{seq}"
        cursor.execute("""
            UPDATE tokens
            SET word_seq = ?,
                align_id = ?
            WHERE id = ?
        """, (seq, align_id, token_id))

    print(f"✅ 已填充 word_seq / align_id 数量：{len(word_tokens)}")

    conn.commit()
    conn.close()

    print("🎉 本章 token 构建完成")


if __name__ == "__main__":
    build_tokens_for_chapter(
        book_abbr="GEN",
        book_id=1,
        chapter=1
    )

📖 处理章节：GEN 1
  需要处理的 verse 数：31
✅ 插入 token 数：1660
🔢 正在填充 word_seq...
✅ 已填充 word_seq / align_id 数量：778
🎉 本章 token 构建完成


In [5]:
import sqlite3
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    """
    按字符类型切分英文文本
    返回 [(token, type), ...]
    """
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_tokens_for_chapter(book_abbr: str, book_id: int, chapter: int):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print(f"📖 处理章节：{book_abbr} {chapter}")

    # 1️⃣ 从 verses 表查询指定 book + chapter
    cursor.execute("""
        SELECT v.id, v.verse, v.text_en
        FROM verses v
        WHERE v.book_id = ?
          AND v.chapter = ?
          AND v.text_en IS NOT NULL
        ORDER BY v.verse
    """, (book_id, chapter))

    verses = cursor.fetchall()
    print(f"  需要处理的 verse 数：{len(verses)}")

    token_rows = []

    # 2️⃣ 遍历 verse → 生成 tokens
    for verse_id, verse_num, text_en in verses:
        tokens = segment_english_preserve(text_en)

        # 校验拼接一致性
        reconstructed = "".join(t for t, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        for idx, (token, token_type) in enumerate(tokens, start=1):
            token_id = f"{book_abbr}.{chapter}.{verse_num}.{idx}"

            token_rows.append((
                token_id,
                book_id,
                chapter,
                verse_num,
                idx,
                token,
                token_type,
                None  # entity_key
            ))

    # 3️⃣ 批量插入 tokens
    cursor.executemany("""
        INSERT INTO tokens (
            id,
            book_id,
            chapter,
            verse,
            token_id,
            token,
            type,
            entity_key
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, token_rows)

    print(f"✅ 插入 token 数：{len(token_rows)}")

    # 4️⃣ 填充 word_seq（✅ 修正排序）
    print("🔢 正在填充 word_seq...")

    cursor.execute("""
        SELECT id
        FROM tokens
        WHERE book_id = ?
          AND chapter = ?
          AND type = 'word'
        ORDER BY chapter, verse, token_id
    """, (book_id, chapter))

    word_tokens = cursor.fetchall()

    for seq, (token_id,) in enumerate(word_tokens, start=1):
        align_id = f"{book_abbr}.{chapter}..{seq}"
        cursor.execute("""
            UPDATE tokens
            SET word_seq = ?,
                align_id = ?
            WHERE id = ?
        """, (seq, align_id, token_id))

    print(f"✅ 已填充 word_seq / align_id 数量：{len(word_tokens)}")

    conn.commit()
    conn.close()

    print("🎉 本章 token 构建完成")


if __name__ == "__main__":
    build_tokens_for_chapter(
        book_abbr="Gen",
        book_id=1,
        chapter=2
    )

📖 处理章节：Gen 2
  需要处理的 verse 数：25
✅ 插入 token 数：1289
🔢 正在填充 word_seq...
✅ 已填充 word_seq / align_id 数量：622
🎉 本章 token 构建完成
